In [ ]:
import pandas as pd



## Import ISC

In [ ]:
PATH = '/media/rsafran/CORSAIR/isc-gem/'
# PATH = '/media/rsafran/CORSAIR/isc-ehb/'
# NAME_ISC = 'isc-ehb.csv'
NAME_ISC = 'isc-gem-cat.csv'
# NAME_ISC = 'USGS_cat.csv' #in ehb file
isc_cnames = ['date','lat','lon','smajax','sminax','strike','q','depth','unc','q.1','mw','unc','q.2','s','mo','fac','mo_auth','mpp','mpr','mrr','mrt','mtp','mtt','str1','dip1','rake1','str2','dip2','rake2','type','eventid']

isc_cat_csv = pd.read_csv(PATH + NAME_ISC, comment='#',header=0,skipinitialspace=True)
isc_cat_csv['date'] = pd.to_datetime(isc_cat_csv['date'],format='ISO8601', utc=True) #Gem
# isc_cat_csv['date'] = pd.to_datetime(isc_cat_csv['time'],format='ISO8601', utc=True) #USGS
# isc_cat_csv.loc[:,'date'] = pd.to_datetime(isc_cat_csv.loc[:,'DATE'] + ' ' + isc_cat_csv.loc[:,'TIME'], format='ISO8601', utc=True)

In [ ]:
isc_cat_csv

In [ ]:
isc_cat_csv.dropna(axis=1, how='any', inplace=True)
isc_cat_csv.rename(columns= {'LAT':'lat', 'LON':'lon','latitude':'lat', 'longitude':'lon', "DEPTH": "depth","MAG":'mw',"mag":'mw', 'EVENTID':'eventid', 'id':'eventid'}, inplace=True)
isc_cat_csv.sort_values('date', inplace=True)


In [ ]:
isc_cat_csv = isc_cat_csv[(isc_cat_csv['date'] > '2016-01-01') & (isc_cat_csv['date'] < '2017-02-01' )]

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import numpy as np
import matplotlib.path as mpltPath
import pandas as pd
from matplotlib.colors import LightSource
import xarray as xr
import os

# ---------------------------------------------------------
# Projection centrée
# ---------------------------------------------------------
# ortho = ccrs.Orthographic(central_longitude=central_lon, central_latitude=central_lat)
pc = ccrs.PlateCarree()
fig, axis = plt.subplots(figsize=(15, 15), subplot_kw=dict(projection=pc))

# ---------------------------------------------------------
# Terres et côtes (couleurs claires pour impression)
# ---------------------------------------------------------
axis.add_feature(cfeature.LAND, facecolor="#f5f5f5", edgecolor="#666666", linewidth=0.5, zorder=2)
axis.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor="#333333", zorder=3)
axis.add_feature(cfeature.BORDERS, linestyle="--", linewidth=0.5, edgecolor="#999999", zorder=3)
axis.set_xlim(-0, 180)
axis.set_ylim(-90, 30)
axis.scatter(
    isc_cat_csv["lon"],
    isc_cat_csv["lat"],
    s=isc_cat_csv["mw"]*2,
    c=isc_cat_csv["depth"],
    marker=".",
    cmap='seismic',
    transform=pc,
    alpha=0.5,
    zorder=4,
)

y = np.array([  [12.280,88.756],
         [-2.742,92.799],
         [-12.829,108.795],
         [-15.047,116.881],
         [-8.859,125.670],
         [6.210,103.698],
         [11.936,91.042],
          ])

y_indian = np.array([[10.260,52.875],
                    # [24.316,67.640],
                    [18.133,89.187],
                    [7.482,110.437],
                    [-7.636,120.578],
                    [-35.729,112.289],
                    [-49.533,114.047],
                    [-58.200,112.640],
                    [-58.751,75.023],
                    [-56.974,45.140],
                    [-58.569,53.929],
                    [-51.043,36.000],
                    [-37.345,32.836],
                    [-27.577,38.812],
                    [-33.577,59.812],
                    # [-28.199,45.468],
                    [-15.846,57.445],
                    [-9.331,54.281],
                    [8.178,51.820],
                    ])

p = Polygon(y_indian[:,::-1], facecolor="orange", edgecolor="purple")

isc_formatted = np.hstack((isc_cat_csv[['lon']].values.flatten()[:,np.newaxis],isc_cat_csv[['lat']].values.flatten()[:,np.newaxis]))
path = mpltPath.Path(y_indian[:,::-1])
inside2 = path.contains_points(isc_formatted)
isc_cat_csv[inside2].plot(x='lon', y='lat', ax=axis, kind='scatter', color='gray', marker='o',  zorder=3)
axis.add_patch(p)
plt.show()

In [ ]:
sumatra_isc = isc_cat_csv[inside2]
print(len(sumatra_isc))

## Load detections

In [ ]:
from utils.data_reading.sound_data.station import StationsCatalog
import pickle
import datetime
import glob2
from pathlib import Path

from utils.physics.sound_model.ellipsoidal_sound_model import GridEllipsoidalSoundModel
from utils.detection.association_geodesic_ridges import compute_candidates, update_valid_grid, update_results, load_detections, compute_grids

import os
from utils.physics.sound_model.ellipsoidal_sound_model import GridEllipsoidalSoundModel

# paths
CATALOG_PATH = "/media/rsafran/CORSAIR/OHASISBIO/recensement_stations_OHASISBIO_RS.csv"  # csv catalog files
DETECTIONS_DIR = "/media/rsafran/CORSAIR/T-pick_2.2"  # where we have detection pickles
ISAS_PATH = "/media/rsafran/CORSAIR/ISAS/extracted/2016"
YEAR = 2016
 # output dir
ASSOCIATIONS_DIR = f"../../data/detection/T-pick_2.2/{YEAR}"
Path(ASSOCIATIONS_DIR).mkdir(exist_ok=True)

# delimitation of the detections we keep (this notebook actually associates the detections of 1 year and 4 hours)
DATE_START = datetime.datetime(YEAR, 1, 1) - datetime.timedelta(hours=2)
DATE_END = datetime.datetime(YEAR+1, 3, 1) + datetime.timedelta(hours=2)

# Detections loading parameters
MIN_P_TISSNET_PRIMARY = 0.5 # min probability of browsed detections
MIN_P_TISSNET_SECONDARY = 0.45 # min probability of detections that can be associated with the browsed one
MERGE_DELTA_S = 1 # threshold below which we consider two events should be merged

# The REQ_CLOSEST_STATIONS th closest stations will be required for an association to be valid
# e.g. if we set it to 6, no association of size <6 will be saved (this is useful to save memory)
REQ_CLOSEST_STATIONS = 6

# sound model definition

arr = os.listdir(ISAS_PATH)
file_list = [os.path.join(ISAS_PATH, fname) for fname in arr if fname.endswith('.nc')]
SOUND_MODEL = GridEllipsoidalSoundModel(file_list)

# association running parameters
SAVE_PATH_ROOT = None  # change this to save the grids as figures, leave at None by default

STATIONS = StationsCatalog(CATALOG_PATH).filter_out_undated().filter_out_unlocated()
# load detection files and keep all the ones that are included in the wanted year
FILES = {}
for f in glob2.glob(f"{DETECTIONS_DIR}/*.pkl"):
    det_files = [f for f in glob2.glob(DETECTIONS_DIR + "/*") if Path(f).is_file()]
    det_files = [f for f in det_files if "2016" in f ]
    dataset, s_name = f[:-4].split("/")[-1].split("_")
    s = STATIONS.by_dataset(dataset).by_name(s_name)
    if len(s) != 1:
        print(f"station {dataset}_{s_name} not found or not unique")
        continue
    FILES[s[0]] = f
FILES = {s : FILES[s] for s in FILES if (s.date_end > DATE_START and s.date_start < DATE_END)}


FILES_2018 = {}
for key in FILES.keys() :
    if key.dataset == "OHASISBIO-2016" or key.dataset == "IMS-2016":
        FILES_2018[key] = FILES[key]
FILES = FILES_2018

In [ ]:
start_times = {}
for s in STATIONS :
    start_times[s.name] = s.date_start

start_times

In [ ]:
drift_ppm ={'H01W1': np.float64(0.0),
             'H08S1': np.float64(0.0),
             'MAD': np.float64(0.1016655380991213),
             'NCRO3': np.float64(-0.11876039356057246),
             'NSPA': np.float64(0.25853181523886887),
             'SSEIR': np.float64(0.14321946805381072),
             'SWAMS1': np.float64(0.09443231135968534),
             'SWAMS3-bot': np.float64(0.009268584899319803),
             'WKER2': np.float64(0.18178731459992636)}

DETECTIONS = load_detections(list(FILES_2018.values()), STATIONS, MIN_P_TISSNET_SECONDARY, merge_delta=datetime.timedelta(seconds=MERGE_DELTA_S))
for s in DETECTIONS.keys():
    print(s.name)
    DETECTIONS[s] = DETECTIONS[s][(DETECTIONS[s][:,0] > DATE_START) & (DETECTIONS[s][:,0] < DATE_END)]

    drift_errors = list(map(lambda t : datetime.timedelta(seconds=s.get_clock_error(t[0],drift_ppm=drift_ppm[s.name])),DETECTIONS[s]))
    DETECTIONS[s][:,0] -= np.array(drift_errors)
# only keep the stations that appear in the kept detections
STATIONS = [s for s in DETECTIONS.keys()]
FIRSTS_DETECTIONS = {s : DETECTIONS[s][0,0] for s in STATIONS}
LASTS_DETECTIONS = {s : DETECTIONS[s][-1,0] for s in STATIONS}


In [ ]:
DETECTIONS[s][:,0]

In [ ]:
s_coords = {}
s_obj = {}
for i,s in enumerate(STATIONS) :
    s_coords[s.name]=s.get_pos()
    s_obj[s.name] = s



In [ ]:
# i = 67474
# travel_time = SOUND_MODEL.get_sound_travel_time(s_coords[name], sumatra_isc[['lat','lon']].loc[i].values,sumatra_isc[['date']].loc[i].dt.to_pydatetime()[0])
# datetime.timedelta(seconds=travel_time)
# print(sumatra_isc[['date']].loc[i])
# sumatra_isc[['date']].loc[i]+datetime.timedelta(seconds=travel_time)
# sumatra_isc

## compute arrivals

In [ ]:
sumatra_final = sumatra_isc.copy()


names = {0: 'MAD',
         1: 'NCRO3',
         2: 'NSPA',
         3: 'SSEIR',
         4: 'SWAMS1',
         5: 'SWAMS3-bot',
         6: 'WKER2'}

for station_id, name in names.items():

    sumatra_final[f'travel_time_{name}'] = sumatra_final.apply(
        lambda row: SOUND_MODEL.get_sound_travel_time(
            s_coords[name],
            row[['lat', 'lon']].values,
            row['date']
        ),
        axis=1
    )

    sumatra_final[f'expected_arrival_{name}'] = (
        sumatra_final['date']
        + pd.to_timedelta(sumatra_final[f'travel_time_{name}'], unit='s')
    )

    # --- détections station ---
    df_det = pd.DataFrame(
        DETECTIONS[s_obj[name]],
        columns=['detection', 'probability']
    )

    df_det['detection'] = pd.to_datetime(df_det['detection'], utc=True)
    df_det = df_det.sort_values('detection')

    # --- merge asof ---
    tmp = sumatra_final[['eventid', f'expected_arrival_{name}']].copy()
    tmp = tmp.sort_values(f'expected_arrival_{name}')

    tmp = pd.merge_asof(
        tmp,
        df_det,
        left_on=f'expected_arrival_{name}',
        right_on='detection',
        tolerance=pd.Timedelta(seconds=5),
        direction='nearest'
    )

    # --- rattachement au dataframe principal ---
    sumatra_final = sumatra_final.merge(
        tmp[['eventid', 'detection']],
        on='eventid',
        how='left'
    )

    sumatra_final.rename(
        columns={'detection': f'detection_{name}'},
        inplace=True
    )

    # --- erreur ---
    sumatra_final[f'diff_{name}'] = (
        sumatra_final[f'expected_arrival_{name}']
        - sumatra_final[f'detection_{name}']
    ).dt.total_seconds()

sumatra_isc

In [ ]:
sumatra_final.plot(x='date',y='diff_WKER2', kind='scatter', legend=False, ax=plt.gca())
# plt.errorbar(x = sumatra_final.date, y = sumatra_final.diff_ELAN, yerr= sumatra_final.rms*10, fmt='+' )

In [ ]:
import numpy as np
import pandas as pd
from numpy.linalg import lstsq, inv, pinv

REF = 'SWAMS-bot'
CORE = [
    'SSEIR',  'RTJ',
    'NEAMS', 'MADW', 'SSWIR','MADE','WKER2','ELAN',
]  # SWAMS-bot retirée

N = len(CORE)
idx = {s: i for i, s in enumerate(CORE)}

# temps en secondes UTC
t_evt = sumatra_final['date'].dt.tz_convert('UTC').astype('int64') / 10**9
t_evt = t_evt.values

t0 = {
    s: int(pd.Timestamp(start_times[s]).timestamp())
    for s in CORE
}


In [ ]:
plt.figure(figsize=(16, 16))

stations = list(names.values())

for i, sta_ref in enumerate(stations):
    plt.subplot(6, 2, i + 1)
    plt.title(sta_ref)

    for sta_cmp in stations:
        if sta_cmp == sta_ref:
            continue

        mask = (
            sumatra_final[f'diff_{sta_ref}'].notna()
            & sumatra_final[f'diff_{sta_cmp}'].notna()
        )

        plt.plot(
            sumatra_final.loc[mask, 'date'],
            sumatra_final.loc[mask, f'diff_{sta_ref}']
            - sumatra_final.loc[mask, f'diff_{sta_cmp}'],
            '+',
            label=sta_cmp
        )

    plt.ylim(-120,120)
    plt.axhline(0, linestyle='--', alpha=0.5)
    plt.legend(framealpha=0.5, ncol=4)

plt.tight_layout()


In [ ]:
plt.figure(figsize=(15, 10))
for i, sta_ref in enumerate(stations):
    plt.subplot(4, 3, i + 1)
    sumatra_final[f'diff_{sta_ref}'].plot(kind='hist',bins = 25,density=True ,histtype='stepfilled',alpha=0.8, label=sta_ref)
    plt.axvline(np.nanmean(sumatra_final[f'diff_{sta_ref}'].values), linestyle='--', alpha=1)
    plt.axvline(np.nanmedian(sumatra_final[f'diff_{sta_ref}'].values), linestyle='--',color='r', alpha=1)
    plt.text(
    0.02, 0.95,
    f"μ={np.nanmean(sumatra_final[f'diff_{sta_ref}'].values):.2f}s\nmed={np.nanmedian(sumatra_final[f'diff_{sta_ref}'].values):.2f}s",
    transform=plt.gca().transAxes,
    va='top'
    )
    plt.legend()

plt.tight_layout()
plt.figure(figsize=(15,10))
stations = list(names.values())
bins = 25
for i, sta_ref in enumerate(stations):
    plt.subplot(4, 3, i + 1)
    plt.title(f"{sta_ref} – erreur relative réseau")

    all_errors = []

    for sta_cmp in stations:
        if sta_cmp == sta_ref:
            continue

        diff = (
            sumatra_final[f'diff_{sta_ref}']
            - sumatra_final[f'diff_{sta_cmp}']
        ).dropna()

        all_errors.append(diff.values)

    all_errors = np.concatenate(all_errors)

    plt.hist(
        all_errors,
        bins=bins,
        density=True,
        histtype='stepfilled',
        alpha=0.8
    )

    plt.axvline(np.nanmedian(all_errors), linestyle='--',color="r", alpha=0.8, label='median')
    plt.axvline(np.nanmean(all_errors), linestyle='--',color="b", alpha=0.8, label='mean')
    plt.text(
        0.02, 0.95,
        f"μ={np.nanmean(all_errors):.2f}s\nmed={np.nanmedian(all_errors):.2f}s",
        transform=plt.gca().transAxes,
        va='top'
    )

    plt.xlabel("Erreur relative (s)")
    plt.ylabel("Densité")

plt.tight_layout()



In [ ]:
from scipy.stats import median_abs_deviation
plt.figure(figsize=(15, 10))
bins = 50

for i, sta_ref in enumerate(stations):
    plt.subplot(4, 3, i + 1)
    plt.title(f"{sta_ref} – erreur relative réseau")

    all_errors = []

    for sta_cmp in stations:
        if sta_cmp == sta_ref:
            continue

        diff = (
            sumatra_final[f'diff_{sta_ref}']
            - sumatra_final[f'diff_{sta_cmp}']
        ).dropna()

        all_errors.append(diff.values)

    all_errors = np.concatenate(all_errors)

    plt.hist(
        all_errors,
        bins=bins,
        density=True,
        histtype='stepfilled',
        alpha=0.8
    )

    mu = np.nanmean(all_errors)
    med = np.nanmedian(all_errors)

    plt.axvline(med, linestyle='--', color='r', label='median')
    plt.axvline(mu, linestyle='--', color='b', label='mean')


    mad = median_abs_deviation(all_errors, scale='normal')

    plt.axvspan(
        np.nanmedian(all_errors) - mad,
        np.nanmedian(all_errors) + mad,
        alpha=0.2,
        label='MAD',
        color='gray',
    )


    plt.text(
        0.02, 0.95,
        f"μ = {mu:.2f} s\nmed = {med:.2f} s\nmad = {mad:.2f} s",
        transform=plt.gca().transAxes,
        va='top'
    )

    plt.xlabel("Erreur relative (s)")
    plt.ylabel("Densité")
    plt.legend()

plt.tight_layout()


## Relative drift

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress
from sklearn.linear_model import TheilSenRegressor

# Entrées
stations = list(names.values())   # liste des 12 stations
target = 'SSEIR'                   # changer pour la station que vous voulez vérifier
bins = 40

# Préparer le vecteur temps en secondes (float), centré pour stabilité numérique
date_series = sumatra_final['date'].dt.tz_convert('UTC')
t_sec = (date_series - date_series.mean()).dt.total_seconds().values  # centrée
t_abs = (date_series).values  # pour l'affichage en x (datetime64[ns, UTC])

results = []

plt.figure(figsize=(14, 18))
plot_idx = 1

for cmp in stations:
    if cmp == target:
        continue

    col_t = f'diff_{target}'
    col_c = f'diff_{cmp}'

    # Mask : événements où les deux stations ont une valeur
    mask = sumatra_final[col_t].notna() & sumatra_final[col_c].notna()
    n_pts = mask.sum()
    if n_pts < 5:
        print(f"Paired {target} vs {cmp}: trop peu de points ({n_pts}), skip.")
        continue

    y = (sumatra_final.loc[mask, col_t] - sumatra_final.loc[mask, col_c]).values  # en secondes
    t = t_sec[mask]  # centrée en secondes
    dates = sumatra_final.loc[mask, 'date']

    # --- OLS (linregress) ---
    ols = linregress(t, y)
    slope_ols = ols.slope            # s / s
    intercept_ols = ols.intercept
    r_value = ols.rvalue
    # calculer erreur standard de la pente
    residuals = y - (slope_ols * t + intercept_ols)
    sigma2 = np.sum(residuals**2) / max(n_pts - 2, 1)
    slope_err = np.sqrt(sigma2 / np.sum((t - np.mean(t))**2))

    # --- Theil-Sen (robuste) ---
    try:
        ts_model = TheilSenRegressor(random_state=0).fit(t.reshape(-1,1), y)
        slope_ts = ts_model.coef_[0]
        intercept_ts = ts_model.intercept_
    except Exception as e:
        slope_ts = np.nan
        intercept_ts = np.nan

    # convertir en ppm
    slope_ols_ppm = slope_ols * 1e6
    slope_ts_ppm = slope_ts * 1e6 if not np.isnan(slope_ts) else np.nan
    slope_err_ppm = slope_err * 1e6

    results.append({
        'pair': f'{target}-{cmp}',
        'n': int(n_pts),
        'slope_ols_s_per_s': slope_ols,
        'slope_ols_err_s_per_s': slope_err,
        'slope_ols_ppm': slope_ols_ppm,
        'slope_ts_s_per_s': slope_ts,
        'slope_ts_ppm': slope_ts_ppm,
        'r_value': r_value
    })

    # --- plotting (scatter + regressions) ---
    plt.subplot(6, 2, plot_idx)
    plt.title(f'{target} vs {cmp} (n={n_pts})')
    plt.scatter(dates, y, s=8, alpha=0.6, label='y = diff_target - diff_cmp')
    # OLS line
    x_plot = (dates - date_series.mean()).dt.total_seconds().values
    y_ols_line = slope_ols * x_plot + intercept_ols
    plt.plot(dates, y_ols_line, '-', label=f'OLS slope={slope_ols_ppm:.2f} ppm', color='C1')
    # Theil-Sen line
    if not np.isnan(slope_ts):
        y_ts_line = slope_ts * x_plot + intercept_ts
        plt.plot(dates, y_ts_line, '--', label=f'TS slope={slope_ts_ppm:.2f} ppm', color='C2')
    plt.axhline(0, linestyle='--', color='k', alpha=0.4)
    plt.ylabel('Erreur relative (s)')
    plt.ylim(-60,60)
    plt.legend(fontsize='small')
    plot_idx += 1

# tidy and show
plt.tight_layout()
plt.show()

# Résumé tabulaire
df_pairs = pd.DataFrame(results).set_index('pair').sort_values('slope_ols_ppm')
print(df_pairs)

# Plot histogram of slopes (ppm) to see dispersion across pairs
plt.figure(figsize=(8,4))
plt.hist(df_pairs['slope_ols_ppm'].dropna(), bins=20)
plt.xlabel('slope OLS (ppm) for target vs other stations')
plt.ylabel('count')
plt.title(f'Distribution des dérives relatives pour {target}')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
N = len(CORE)
idx = {s: i for i, s in enumerate(CORE)}
MAX_REL_ERR = 20 # secondes


A_rows = []
b_vals = []

for k, t in enumerate(t_evt):
    row = sumatra_final.iloc[k]

    for i, si in enumerate(CORE):
        vi = row.get(f'diff_{si}', np.nan)
        if pd.isna(vi):
            continue

        vref = row.get(f'diff_{REF}', np.nan)
        if not pd.isna(vref):
            b = vi - vref
            if abs(b) <= MAX_REL_ERR:
                arow = np.zeros(N)
                arow[i] = (t - t0[si])
                A_rows.append(arow)
                b_vals.append(b)

        for j in range(i + 1, N):
            sj = CORE[j]
            vj = row.get(f'diff_{sj}', np.nan)
            if pd.isna(vj):
                continue

            b = vi - vj
            if abs(b) <= MAX_REL_ERR:
                arow = np.zeros(N)
                arow[i] = (t - t0[si])
                arow[j] = -(t - t0[sj])
                A_rows.append(arow)
                b_vals.append(b)


A = np.vstack(A_rows)
b = np.array(b_vals)


In [ ]:
print("Nombre d'équations :", len(b_vals))
print("max |b| :", np.max(np.abs(b_vals)))
print("median |b| :", np.median(np.abs(b_vals)))


In [ ]:
def huber_weights(r, sigma, c=0.345):
    z = np.abs(r / sigma)
    w = np.ones_like(z)
    mask = z > c
    w[mask] = c / z[mask]
    return w


In [ ]:
def solve_ls(A, b, W=None):
    if W is None:
        return np.linalg.lstsq(A, b, rcond=None)[0]
    w = np.sqrt(W)
    return np.linalg.lstsq(A * w[:, None], b * w, rcond=None)[0]

x = solve_ls(A, b)
W = np.ones(len(b))

for _ in range(10000):
    r = A @ x - b
    sigma = 1.4826 * np.median(np.abs(r - np.median(r)))
    if sigma == 0:
        break
    W_new = huber_weights(r, sigma)
    x_new = solve_ls(A, b, W=W_new)
    if np.linalg.norm(x_new - x) < 1e-12:
        break
    x, W = x_new, W_new


In [ ]:
df_core = pd.DataFrame({
    'station': CORE,
    'drift_ppm': x * 1e6
}).set_index('station')

df_core.loc[REF] = 0.0
print(df_core)


In [ ]:
names